In [22]:
import os
import pandas as pd
from datetime import datetime

file_data = []
for name in os.listdir('.'):
    info = os.stat(name)
    file_data.append({
        "파일명": name,
        "크기(Bytes)": info.st_size,
        "수정일시": datetime.fromtimestamp(info.st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
        "구분": "폴더" if os.path.isdir(name) else "파일"
    })

# 데이터프레임으로 변환해서 출력
df = pd.DataFrame(file_data)
df

,파일명,크기(Bytes),수정일시,구분
0,.bashrc,3823,2023-10-20 01:46:06,파일
1,.profile,807,2022-01-06 16:23:33,파일
2,.bash_logout,220,2022-01-06 16:23:33,파일
3,tencafe_20260615004854.csv,116422,2026-06-15 00:48:54,파일
4,google-chrome-stable_current_amd64.deb.3,130424340,2026-06-11 19:02:56,파일
5,.npm,4096,2023-10-20 01:50:11,폴더
6,dataset,4096,2026-06-12 04:17:26,폴더
7,.jupyter,4096,2026-06-15 00:13:52,폴더
8,google-chrome-stable_current_amd64.deb.1,130424340,2026-06-11 19:02:56,파일
9,google-chrome-stable_current_amd64.deb.5,130424340,2026-06-11 19:02:56,파일


In [23]:
%pip install selenium beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


In [24]:
%pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [25]:
%pip install selenium beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


In [26]:
import requests
from bs4 import BeautifulSoup as bs

# 오피넷 내부에서 실제 지역별 데이터를 불러오는 가짜 API 주소입니다.
url = "https://www.opinet.co.kr/searRgSelect.do"

# 크롬 브라우저인 척 오피넷 서버를 속이는 가짜 헤더 값 설정
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Origin": "https://www.opinet.co.kr",
    "Referer": "https://www.opinet.co.kr/searRgSelect.do"
}

# 부산광역시(SIDO_NM0)를 요청할 때 오피넷 서버에 보낼 파라미터 양식
payload = {
    "SIDO_NM0": "부산광역시",
    "SIGUNGU_NM0": "", # 구를 비워두면 전체 조회 효과가 나거나 첫 번째 구가 잡힙니다.
    "SIDO_CD": "02"     # 오피넷 내부 부산광역시 코드값 (예시)
}

print("크롬 브라우저 없이 오피넷 서버와 직접 통신을 시도합니다...")
response = requests.post(url, headers=headers, data=payload)

if response.status_code == 200:
    print("✅ 서버 통신 성공! 데이터를 파싱합니다.")
    soup = bs(response.text, 'html.parser')
    
    # 주유소 리스트 추출
    contents = soup.select('#body1 > tr')
    temp = []
    
    for c in contents:
        title_element = c.select_one('a') 
        if title_element:
            store_name = title_element.text.strip()
            temp.append(store_name)
            
    print(f"\n찾은 주유소 개수: {len(temp)}개")
    print(temp)
else:
    print(f"❌ 오피넷 서버가 접근을 거부했습니다. 상태코드: {response.status_code}")

크롬 브라우저 없이 오피넷 서버와 직접 통신을 시도합니다...
✅ 서버 통신 성공! 데이터를 파싱합니다.

찾은 주유소 개수: 25개
['내트럭(주)부산사...', '내트럭(주)부산용...', '주식회사 팔성', '지피에스㈜ 사랑드...', '매일주유소', '㈜문화주유소', '대일주유소', '지에스칼텍스㈜ 유...', '금강석유(주) 영...', '금융단지주유소', '성지셀프주유소', '분포셀프주유소', '(주)경인석유', '문현주유소', '동명주유소', '반도주유소', '우암로셀프주유소', '은마석유㈜ 용당C...', '북항대교주유소', '부산항 주유소', '우암주유소', '은마석유주식회사', '구도일주유소용호', '대성주유소', '인터지스(주)인터...']


In [27]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd

# 1. 오피넷 지역별 주유소 찾기 실제 데이터 요청 주소
url = "https://www.opinet.co.kr/searRgSelect.do"

# 2. 리눅스 서버 환경에서 봇(Bot)으로 차단당하지 않기 위한 브라우저 우회 헤더 세팅
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Origin": "https://www.opinet.co.kr",
    "Referer": "https://www.opinet.co.kr/searRgSelect.do"
}

# 3. 오피넷 서버에 보낼 지역 선택 데이터 (예시: 부산광역시 전체 조회)
payload = {
    "SIDO_NM0": "부산광역시",
    "SIGUNGU_NM0": "",   # 특정 구를 넣거나 비워두면 전체 혹은 기본 구가 조회됩니다.
    "SIDO_CD": "02"       # 오피넷 내부 부산광역시 행정 코드
}

print("🚀 브라우저 없이 오피넷 서버와 직접 통신을 시도합니다...")

try:
    # 서버에 POST 방식으로 데이터 요청
    response = requests.post(url, headers=headers, data=payload, timeout=10)
    
    if response.status_code == 200:
        print("✅ 오피넷 데이터 수집 성공! 파싱을 시작합니다.")
        
        # BeautifulSoup으로 받아온 HTML HTML 분석
        soup = bs(response.text, 'html.parser')
        
        # 1. 원래 확인하고 싶어 하셨던 [구/군 드롭다운 옵션 정보] 미리 파싱하기
        sigungu_options = soup.select('#SIGUNGU_NM0 > option')
        # '선택' 같은 기본 문구를 제외한 실제 구/군 이름만 필터링
        district_list = [opt.text.strip() for opt in sigungu_options if opt.text.strip() and '선택' not in opt.text]
        
        print(f"\n📊 [구/군 조회 결과] 총 {len(district_list)}개의 구/군을 찾았습니다.")
        print(f"구/군 리스트: {district_list}\n")
        print("-" * 50)

        # 2. 결과 테이블에서 주유소 리스트 추출
        gas_stations = soup.select('#body1 > tr')
        station_names = []
        
        for station in gas_stations:
            name_element = station.select_one('a')
            if name_element:
                name = name_element.text.strip()
                station_names.append(name)
                print(f"✨ 주유소 발견: {name}")
                
        print("-" * 50)
        print(f"🎉 [최종 결과] 수집 완료된 주유소 총 {len(station_names)}개")
        print(station_names)
        
    else:
        print(f"❌ 오피넷 서버 연결 실패 (상태 코드: {response.status_code})")

except Exception as e:
    print(f"❌ 데이터 수집 중 에러 발생: {e}")

🚀 브라우저 없이 오피넷 서버와 직접 통신을 시도합니다...
✅ 오피넷 데이터 수집 성공! 파싱을 시작합니다.

📊 [구/군 조회 결과] 총 17개의 구/군을 찾았습니다.
구/군 리스트: ['시/군/구', '강서구', '금정구', '기장군', '남구', '동구', '동래구', '부산진구', '북구', '사상구', '사하구', '서구', '수영구', '연제구', '영도구', '중구', '해운대구']

--------------------------------------------------
✨ 주유소 발견: 내트럭(주)부산사...
✨ 주유소 발견: 내트럭(주)부산용...
✨ 주유소 발견: 주식회사 팔성
✨ 주유소 발견: 지피에스㈜ 사랑드...
✨ 주유소 발견: 매일주유소
✨ 주유소 발견: ㈜문화주유소
✨ 주유소 발견: 금강석유(주) 영...
✨ 주유소 발견: 지에스칼텍스㈜ 유...
✨ 주유소 발견: 대일주유소
✨ 주유소 발견: 분포셀프주유소
✨ 주유소 발견: 동명주유소
✨ 주유소 발견: (주)경인석유
✨ 주유소 발견: 문현주유소
✨ 주유소 발견: 금융단지주유소
✨ 주유소 발견: 성지셀프주유소
✨ 주유소 발견: 구도일주유소용호
✨ 주유소 발견: 은마석유㈜ 용당C...
✨ 주유소 발견: 은마석유주식회사
✨ 주유소 발견: 반도주유소
✨ 주유소 발견: 부산항 주유소
✨ 주유소 발견: 북항대교주유소
✨ 주유소 발견: 우암로셀프주유소
✨ 주유소 발견: 우암주유소
✨ 주유소 발견: 대성주유소
✨ 주유소 발견: 인터지스(주)인터...
--------------------------------------------------
🎉 [최종 결과] 수집 완료된 주유소 총 25개
['내트럭(주)부산사...', '내트럭(주)부산용...', '주식회사 팔성', '지피에스㈜ 사랑드...', '매일주유소', '㈜문화주유소', '금강석유(주) 영...', '지에스칼텍스㈜ 유...', '대일주유소', '분포셀프주유소', '동명주유소', '(주)경인석유', '문현주유소', '금융단지주유소', '성

In [28]:
!pip show selenium

Name: selenium
Version: 4.44.0
Summary: Official Python bindings for Selenium WebDriver
Home-page: https://www.selenium.dev
Author: 
Author-email: 
License: Apache-2.0
Location: /opt/conda/lib/python3.11/site-packages
Requires: certifi, trio, trio-websocket, typing_extensions, urllib3, websocket-client
Required-by: 


In [29]:
!pip show webdriver-manager

Name: webdriver-manager
Version: 4.1.2
Summary: Library provides the way to automatically manage drivers for different browsers
Home-page: 
Author: 
Author-email: Sergey Pirogov <automationremarks@gmail.com>
License: 
Location: /opt/conda/lib/python3.11/site-packages
Requires: packaging, python-dotenv, requests
Required-by: 


In [30]:
# !which google-chrome
# !which chromium-browser
!google-chrome --version

/bin/bash: line 1: google-chrome: command not found


In [3]:
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

url = "https://www.opinet.co.kr/user/main/mainView.do"
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

print("✅ 접속 성공!")
print("페이지 제목:", soup.title.text)

✅ 접속 성공!
페이지 제목: 싼 주유소 찾기 오피넷


In [12]:
import time as tt
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.add_experimental_option("detach", True)
options.binary_location = r"C:\Program Files\Google\Chrome\Application\chrome.exe"

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

driver.get("https://www.opinet.co.kr/user/main/mainView.do")
tt.sleep(10)  # ✅ 대기 시간 늘림

# ✅ 요소가 있는지 먼저 테스트
try:
    element = driver.find_element(By.ID, "SIGUNGU_NM0")
    print("✅ SIGUNGU_NM0 요소 찾음!")
except:
    print("❌ SIGUNGU_NM0 요소 없음 → 다른 페이지로 이동 필요!")
    print("현재 페이지 URL:", driver.current_url)
    print("현재 페이지 제목:", driver.title)

❌ SIGUNGU_NM0 요소 없음 → 다른 페이지로 이동 필요!
현재 페이지 URL: https://www.opinet.co.kr/user/main/mainView.do
현재 페이지 제목: 싼 주유소 찾기 오피넷


In [11]:
import time as tt
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.binary_location = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
options.add_experimental_option("detach", True)
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

wait = WebDriverWait(driver, 10)

driver.get("https://www.opinet.co.kr/user/main/mainView.do")
tt.sleep(3)

# ✅ Step 1. 페이지에 iframe이 몇 개 있는지 확인
iframes = driver.find_elements(By.TAG_NAME, "iframe")
print(f"총 iframe 개수: {len(iframes)}개")
for i, frame in enumerate(iframes):
    print(f"  {i}번 iframe | id={frame.get_attribute('id')} | name={frame.get_attribute('name')} | src={frame.get_attribute('src')}")

# ✅ Step 2. 각 iframe에 진입해서 SIGUNGU_NM0 찾기
for i, frame in enumerate(iframes):
    try:
        driver.switch_to.frame(frame)   # iframe 진입
        driver.find_element(By.ID, "SIGUNGU_NM0")
        print(f"\n✅ {i}번 iframe 안에 SIGUNGU_NM0 있음!")
        break
    except:
        print(f"  {i}번 iframe → 없음")
        driver.switch_to.default_content()  # 메인으로 복귀

총 iframe 개수: 2개
  0번 iframe | id=frogue-chat-iframe | name=frogue-chat-iframe | src=https://frogue.danbee.ai/?chatbot_id=a131b207-2fb0-4c60-9cab-873274fb2e2a&user_id=jungeun.kim@knoc.co.kr
  1번 iframe | id=frogue-btn-iframe | name=frogue-btn-iframe | src=https://frogue.danbee.ai/button/?chatbot_id=a131b207-2fb0-4c60-9cab-873274fb2e2a
  0번 iframe → 없음
  1번 iframe → 없음


In [1]:
import time as tt
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.binary_location = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
options.add_experimental_option("detach", True)
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

driver.get("https://www.opinet.co.kr/user/main/mainView.do")
tt.sleep(5)  # ✅ 넉넉하게 대기

# ✅ 페이지의 모든 select 태그 찾기
selects = driver.find_elements(By.TAG_NAME, "select")
print(f"📋 페이지의 select 태그 총 {len(selects)}개\n")
print("-" * 50)

for i, s in enumerate(selects):
    sid = s.get_attribute('id')
    name = s.get_attribute('name')
    options_list = [o.text for o in s.find_elements(By.TAG_NAME, "option")][:5]  # 앞 5개만
    print(f"{i}번 | id='{sid}' | name='{name}'")
    print(f"     옵션 예시: {options_list}")
    print()

print("-" * 50)
print("✅ 위 목록에서 시/도 관련 select ID를 찾아보세요!")

📋 페이지의 select 태그 총 2개

--------------------------------------------------
0번 | id='selected1' | name=''
     옵션 예시: ['전체', '강서구', '금정구', '기장군', '남구']

1번 | id='selected2' | name=''
     옵션 예시: ['저가순', '고가순']

--------------------------------------------------
✅ 위 목록에서 시/도 관련 select ID를 찾아보세요!


In [2]:
# 버튼이 뭔지 먼저 찾기
import time as tt
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.binary_location = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
options.add_experimental_option("detach", True)
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

driver.get("https://www.opinet.co.kr/user/main/mainView.do")
tt.sleep(3)

# ✅ 페이지의 모든 버튼/링크 찾기
print("📋 페이지의 버튼 목록:")
btns = driver.find_elements(By.TAG_NAME, "button")
for i, b in enumerate(btns):
    print(f"  버튼 {i}번 | text='{b.text}' | id='{b.get_attribute('id')}'")

print("\n📋 페이지의 a 태그(링크) 목록:")
links = driver.find_elements(By.TAG_NAME, "a")
for i, a in enumerate(links):
    if a.text.strip():  # 텍스트 있는 것만
        print(f"  링크 {i}번 | text='{a.text.strip()}' | id='{a.get_attribute('id')}'")

📋 페이지의 버튼 목록:
  버튼 0번 | text='' | id=''
  버튼 1번 | text='' | id=''
  버튼 2번 | text='' | id=''
  버튼 3번 | text='' | id=''

📋 페이지의 a 태그(링크) 목록:
  링크 0번 | text='본문 내용 바로가기' | id=''
  링크 2번 | text='싼 주유소찾기' | id=''
  링크 9번 | text='국내유가통계' | id=''
  링크 22번 | text='유가관련정보' | id=''
  링크 32번 | text='불법행위공표' | id=''
  링크 35번 | text='이용안내' | id=''
  링크 43번 | text='로그인' | id=''
  링크 44번 | text='회원가입' | id=''
  링크 45번 | text='휘발유' | id='tab_1'
  링크 46번 | text='경유' | id='tab_2'
  링크 47번 | text='LPG' | id='tab_3'
  링크 48번 | text='1주' | id='chart_w'
  링크 49번 | text='1개월' | id='chart_m'
  링크 50번 | text='1년' | id='chart_y'
  링크 51번 | text='3년' | id='chart_t'
  링크 52번 | text='내트럭(주)부산사업소' | id=''
  링크 53번 | text='내트럭(주)부산용당' | id=''
  링크 54번 | text='주식회사 팔성' | id=''
  링크 55번 | text='매일주유소' | id=''
  링크 56번 | text='지피에스㈜ 사랑드림주유소' | id=''
  링크 57번 | text='㈜씨더블유 이엔지 금천지점' | id=''
  링크 58번 | text='삼양주유소' | id=''
  링크 59번 | text='대일종합에너지주유소 태전점' | id=''
  링크 60번 | text='차오름에너지㈜' | id=''
  링크 61번 | text='(주)평동제일

In [14]:
import time as tt
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.binary_location = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
options.add_experimental_option("detach", True)
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
wait_long = WebDriverWait(driver, 15)

driver.get("https://www.opinet.co.kr/user/main/mainView.do")
tt.sleep(4)

# ✅ selected1 드롭다운 옵션 개수 확인
sigungu_element = wait_long.until(EC.presence_of_element_located((By.ID, "selected1")))
sigungu_select  = Select(sigungu_element)
option_count    = len(sigungu_select.options)
print(f"✅ 총 구/군 개수: {option_count - 1}개")

results = []

print("\n" + "=" * 45)
print("        데이터 수집 시작!")
print("=" * 45)

try:
    for i in range(1, option_count):

        # 드롭다운 새로 찾기
        sigungu_element = wait_long.until(EC.presence_of_element_located((By.ID, "selected1")))
        sigungu_select  = Select(sigungu_element)
        sigungu_select.select_by_index(i)
        selected_name   = sigungu_select.first_selected_option.text
        print(f"\n[{i}/{option_count-1}] {selected_name} 수집 중...")
        tt.sleep(3)  # 목록 갱신 대기

        try:
            # ✅ tbody 안의 tr 태그로 주유소 개수 파악
            tbodies = driver.find_elements(By.TAG_NAME, "tbody")

            station_count = 0
            station_names = []

            for tb in tbodies:
                rows = tb.find_elements(By.TAG_NAME, "tr")
                for row in rows:
                    text = row.text.strip()
                    # 빈 행, 헤더 제외
                    if text and "검색된 결과" not in text and len(text) > 5:
                        station_count += 1
                        station_names.append(text[:40])

            print(f"  ✅ {selected_name} → {station_count}개 주유소 발견!")

            # 상위 3개 주유소 미리보기
            for name in station_names[:3]:
                print(f"    - {name}")

            results.append({
                "순번": i,
                "지역": selected_name,
                "주유소수": station_count,
                "상태": "✅ 완료"
            })

        except Exception as e:
            print(f"  ❌ 파싱 실패: {e}")
            results.append({"순번": i, "지역": selected_name, "주유소수": 0, "상태": "❌ 실패"})

        tt.sleep(1)

except Exception as e:
    print(f"\n❌ 에러 발생: {e}")

finally:
    print("\n" + "=" * 45)
    print("           수집 결과 요약")
    print("=" * 45)
    print(f"  {'순번':<6} {'지역':<12} {'주유소수':<8} {'상태'}")
    print("-" * 40)
    for r in results:
        print(f"  {r['순번']:<6} {r['지역']:<12} {r['주유소수']}개{'':<4} {r['상태']}")
    print("-" * 40)
    total = sum(r['주유소수'] for r in results)
    print(f"  총 주유소 수: {total}개")
    print("=" * 45)

✅ 총 구/군 개수: 16개

        데이터 수집 시작!

[1/16] 강서구 수집 중...
  ✅ 강서구 → 22개 주유소 발견!
    - 공항대로주유소 1,965
    - 동방석유㈜직영 대저주유소 1,968
    - 서강주유소 1,968

[2/16] 금정구 수집 중...
  ✅ 금정구 → 22개 주유소 발견!
    - 한길주유소 금사점 한솔유화㈜ 1,979
    - ㈜진진에너지 1,979
    - 은마석유 노포주유소 1,984

[3/16] 기장군 수집 중...
  ✅ 기장군 → 22개 주유소 발견!
    - 신아시아드주유소/충전소 은마석유㈜ 1,955
    - 월드컵주유소 1,955
    - 기장대로 셀프주유소 와이케이에너지㈜ 1,955

[4/16] 남구 수집 중...
  ✅ 남구 → 22개 주유소 발견!
    - 내트럭(주)부산사업소 1,969
    - 내트럭(주)부산용당 1,969
    - 주식회사 팔성 1,979

[5/16] 동구 수집 중...
  ✅ 동구 → 22개 주유소 발견!
    - 시원석유㈜해수부주유소 1,969
    - 동방석유(주)초량주유소 1,969
    - 청구주유소 1,975

[6/16] 동래구 수집 중...
  ✅ 동래구 → 22개 주유소 발견!
    - 대경주유소 1,968
    - 광신석유(주)직영 충렬대로주유소 1,969
    - ㈜OS에너지 동호주유소 1,969

[7/16] 부산진구 수집 중...
  ✅ 부산진구 → 22개 주유소 발견!
    - 대양주유소 1,963
    - 진성3주유소 1,963
    - 백양주유소 1,965

[8/16] 북구 수집 중...
  ✅ 북구 → 22개 주유소 발견!
    - 광신석유(주)직영 백양대로주유소 1,965
    - 경덕주유소 1,968
    - 화명신도시주유소 1,975

[9/16] 사상구 수집 중...
  ✅ 사상구 → 22개 주유소 발견!
    - 우리주유소 (주)시블링스 1,962
    - (주)한영 주례지점 